# MMDP-OD-RTDP Comparison — Final Experiment

The notebook supports three ways to load the project:

1. **GitHub** — clone the current repository automatically.
2. **Manual folder** — use a complete project folder already uploaded or copied into Colab.
3. **Manual ZIP** — upload a ZIP containing the complete project folder and extract it automatically.

Choose the source in the preparation cell below. A valid project folder must contain
`pyproject.toml`, `src/mmdp`, `scripts`, and `maps`.

The experiment itself is fixed and requires no parameter choices:

- one map per difficulty level
- 1–6 agents
- Baseline RTDP vs OD-RTDP
- one fixed reproducible seed: `20260708`
- planning up to 60 seconds or until `solved`
- evaluation of up to 5 episodes, with a maximum budget of 8 seconds
- no evaluation diagnostics and no evaluation conflict-risk calculations
- transition cache bounded identically for both algorithms
- no manifest files: the final configuration is defined directly in the package

All results are saved to a single file:

`/content/MMDP_OUTPUT/MMDP_results.csv`

Each difficulty contains 12 conditions: 6 agent counts × 2 algorithms.
The CSV contains the four key metrics—time, memory, real states examined, and
success rate—together with basic run identity and status fields. Nothing is
downloaded automatically.


In [ ]:
# Choose how to load the project, then run this cell.
# In Google Colab these lines appear as editable form fields.
SOURCE_MODE = "github"  # @param ["github", "manual_folder", "manual_zip"]
GITHUB_REPOSITORY = "https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git"  # @param {type:"string"}
MANUAL_PROJECT_PATH = "/content/MMDP-OD-RTDP-PROJECT-CONSOLIDATED"  # @param {type:"string"}

import importlib
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def is_project_root(path: Path) -> bool:
    """Return True when path contains the required project structure."""
    return (
        path.is_dir()
        and (path / "pyproject.toml").is_file()
        and (path / "src" / "mmdp").is_dir()
        and (path / "scripts" / "run_compact_matrix.py").is_file()
        and (path / "maps").is_dir()
    )


def find_project_root(search_root: Path) -> Path:
    """Find the project root itself or inside one uploaded/extracted directory."""
    search_root = search_root.expanduser().resolve()
    if is_project_root(search_root):
        return search_root
    if not search_root.exists():
        raise FileNotFoundError(f"Project path does not exist: {search_root}")

    candidates = []
    for pyproject in search_root.rglob("pyproject.toml"):
        candidate = pyproject.parent
        if is_project_root(candidate):
            candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            "No valid project folder was found. The folder must contain "
            "pyproject.toml, src/mmdp, scripts/run_compact_matrix.py, and maps."
        )
    if len(candidates) > 1:
        listed = "\n".join(f"- {path}" for path in candidates)
        raise RuntimeError(
            "More than one valid project folder was found. Set MANUAL_PROJECT_PATH "
            f"to the required folder:\n{listed}"
        )
    return candidates[0]


def safely_extract_zip(zip_path: Path, destination: Path) -> None:
    """Extract a ZIP while rejecting paths that escape the destination."""
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (destination / member.filename).resolve()
            if destination != member_path and destination not in member_path.parents:
                raise ValueError(f"Unsafe ZIP entry: {member.filename}")
        archive.extractall(destination)


if SOURCE_MODE == "github":
    repo_dir = Path("/content/MMDP-OD-RTDP-PROJECT")
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    print(f"Cloning repository from {GITHUB_REPOSITORY} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", GITHUB_REPOSITORY, str(repo_dir)],
        check=True,
    )
    PROJECT_ROOT = find_project_root(repo_dir)

elif SOURCE_MODE == "manual_folder":
    # First upload/copy the complete folder into Colab using the Files panel,
    # Google Drive, or another method; then set MANUAL_PROJECT_PATH above.
    PROJECT_ROOT = find_project_root(Path(MANUAL_PROJECT_PATH))
    print(f"Using manually supplied project folder: {PROJECT_ROOT}")

elif SOURCE_MODE == "manual_zip":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "manual_zip uses the Google Colab upload dialog. Outside Colab, "
            "extract the ZIP yourself and choose manual_folder."
        ) from exc

    print("Choose one ZIP file containing the complete project folder.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one .zip file.")

    upload_root = Path("/content/MMDP_MANUAL_UPLOAD")
    if upload_root.exists():
        shutil.rmtree(upload_root)
    upload_root.mkdir(parents=True)

    zip_path = Path("/content") / zip_names[0]
    # files.upload normally writes into the current directory. This fallback
    # handles notebooks whose current directory is not /content.
    if not zip_path.exists():
        zip_path = Path(zip_names[0]).resolve()
    safely_extract_zip(zip_path, upload_root)
    PROJECT_ROOT = find_project_root(upload_root)
    print(f"Using project extracted from ZIP: {PROJECT_ROOT}")

else:
    raise ValueError(
        "SOURCE_MODE must be 'github', 'manual_folder', or 'manual_zip'."
    )

OUTPUT_DIR = Path("/content/MMDP_OUTPUT")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / "MMDP_results.csv"

# Remove modules from a previous source selection before reinstalling.
for module_name in list(sys.modules):
    if module_name == "mmdp" or module_name.startswith("mmdp."):
        del sys.modules[module_name]

print("Installing the mmdp package from:", PROJECT_ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", str(PROJECT_ROOT) + "[notebooks]"],
    check=True,
)
importlib.invalidate_caches()

print("Preparation complete.")
print("SOURCE_MODE:", SOURCE_MODE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV:", RESULTS_CSV)


In [ ]:
# Import the map visualization helper from the installed package
from functools import partial

from ipywidgets import interact, IntSlider
from mmdp.analysis.notebook_visualizer import plot_map_visualization

plot_map_visualization = partial(plot_map_visualization, maps_root=PROJECT_ROOT / 'maps')

In [ ]:
# Define the run and analysis functions
import subprocess, sys
import pandas as pd
from IPython.display import display, Image, Markdown
from mmdp.experiments.final_config import FIXED_SEED

GROUP_MAPS = {
    'easy': ['empty-8-8'],
    'medium': ['warehouse-10-20-10-2-1'],
    'hard': ['room-64-64-16'],
}


def run_group(group):
    print('='*80)
    print(f"Starting the {group} experiment: {', '.join(GROUP_MAPS[group])}")
    print('12 conditions: 1 map x 6 agent counts x 2 algorithms')
    print(f'Fixed seed: {FIXED_SEED}')
    print('='*80, flush=True)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT/'scripts/run_compact_matrix.py'),
        '--group', group, '--output', str(RESULTS_CSV),
    ]
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Experiment failed with code {code}')


def comparison_table(group):
    df = pd.read_csv(RESULTS_CSV)
    part = df[(df['map_group'] == group) & (df['status'] == 'ok')].copy()
    metrics = [
        'planning_time_seconds',
        'planning_peak_memory_delta_mb',
        'states_examined',
        'success_rate',
    ]
    for col in ['n_agents', *metrics]:
        part[col] = pd.to_numeric(part[col], errors='coerce')

    duplicated = part.duplicated(['n_agents', 'algorithm'], keep=False)
    if duplicated.any():
        raise ValueError(
            'Expected one fixed-seed row per condition. '
            'Use a new CSV path or remove results from an older experiment.'
        )

    agents = sorted(part['n_agents'].dropna().astype(int).unique())
    result = pd.DataFrame({'n_agents': agents}).set_index('n_agents')
    names = {
        'planning_time_seconds': 'time_s',
        'planning_peak_memory_delta_mb': 'memory_mb',
        'states_examined': 'states',
        'success_rate': 'success',
    }
    for metric, short in names.items():
        for algorithm in ('baseline', 'od'):
            result[f'{short}_{algorithm}'] = (
                part[part['algorithm'] == algorithm]
                .set_index('n_agents')[metric]
                .reindex(agents)
            )
        result[f'{short}_difference_OD_minus_Baseline'] = (
            result[f'{short}_od'] - result[f'{short}_baseline']
        )
    return result.reset_index().round(4)


def analyze_group(group):
    graph_dir = OUTPUT_DIR/'graphs'/group
    command = [
        sys.executable, str(PROJECT_ROOT/'scripts/analyze_compact_results.py'),
        str(RESULTS_CSV), '--group', group, '--output-dir', str(graph_dir),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    display(Markdown('## Fixed-seed comparison table by number of agents'))
    display(comparison_table(group))
    figures = [
        ('01_time.png', 'Planning time - lower is better'),
        ('02_memory.png', 'Planning memory increase - lower is better'),
        ('03_states_and_success.png', 'States examined and success rate'),
    ]
    for filename, title in figures:
        display(Markdown(f'### {title}'))
        display(Image(filename=str(graph_dir/filename)))
    print('\nSingle CSV:', RESULTS_CSV)
    print('Graphs:', graph_dir)
    print('Nothing is downloaded automatically; use the Files panel to download manually.')


def run_and_analyze(group):
    run_group(group)
    analyze_group(group)

print('Preparation finished. Run one of the three difficulty cells below.')


## Easy map

Map: `empty-8-8`.

In [ ]:
def _plot_easy(num_agents):
    plot_map_visualization('empty-8-8', 'empty-8-8-bundled-1', num_agents)
interact(_plot_easy, num_agents=IntSlider(min=1, max=6, step=1, value=3, description='Agents:'))

In [ ]:
run_and_analyze("easy")

## Medium map

Map: `warehouse-10-20-10-2-1`.

In [ ]:
def _plot_med(num_agents):
    plot_map_visualization('warehouse-10-20-10-2-1', 'warehouse-10-20-10-2-1-bundled-1', num_agents)
interact(_plot_med, num_agents=IntSlider(min=1, max=6, step=1, value=4, description='Agents:'))

In [ ]:
run_and_analyze("medium")

## Hard map

Map: `room-64-64-16`.

In [ ]:
def _plot_hard(num_agents):
    plot_map_visualization('room-64-64-16', 'room-64-64-16-even-1', num_agents)
interact(_plot_hard, num_agents=IntSlider(min=1, max=6, step=1, value=6, description='Agents:'))

In [ ]:
run_and_analyze("hard")

## Output

The folder `/content/MMDP_OUTPUT` will contain:

- `MMDP_results.csv` — the single fixed-seed results file from every group that was run
- `graphs/easy` — three figures for the easy level
- `graphs/medium` — three figures for the medium level
- `graphs/hard` — three figures for the hard level

Re-running the same cell skips conditions that already exist in the CSV.
Because the compact CSV schema changed from the earlier multi-seed experiment,
use a new output folder or delete the old CSV before running this version.
